# Etna Dataset Construction

This notebook builds the canonical hourly Etna dataset used by the Cause–Trigger analysis. The saved CSV remains in physical or proxy units; transformations and standardization are applied later to the selected reference and case intervals.

In [1]:
import sys
from pathlib import Path
from obspy.clients.fdsn import Client
from obspy.clients.fdsn import Client
from obspy.io.mseed import InternalMSEEDWarning
import warnings
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src/etna"
sys.path.append(str(SRC_DIR))

from etna_config import (
    EVENT_TIME,
    ETNA_GAS_METEO_COLS,
    ETNA_WEATHER_COLS,
    ETNA_WAVEFORM_CONFIG,
    etna_observable_metadata,
)
from etna_dataset import (
    create_etna_dataset,
    load_etna_event_catalog_xls,
    load_etnagas_csv,
    load_openmeteo_etna_weather,
)
from etna_plotting_utils import (
    dataset_health_report,
    distribution_summary,
    plot_etna_all_variables_map,
    plot_etna_thesis_figures,
    run_teleseismic_checks,
    plot_etna_loglog_distributions,
)
from etna_waveform import build_station_waveform_dataset

warnings.filterwarnings(
    "ignore",
    category=InternalMSEEDWarning,
    message=r".*fractional second.*10000.*",
)

### Etna seismic-station screening

Candidate vertical seismic streams within 30 km of Etna were screened using short daily waveform probes. Streams were ranked by data availability, distance from the summit, and channel type. Based on this screening, IV.ESLN..HHZ was selected for constructing the teleseismic proxy.

In [2]:
# This is not required for routine dataset reconstruction 
from etna_stations import (
    export_station_screening_results,
    run_station_screening,
)

# True to run station screening, False to skip and use existing results
RUN_STATION_SCREENING = True

if RUN_STATION_SCREENING:
    (
        candidate_streams,
        station_daily_probe,
        station_summary,
    ) = run_station_screening(
        client=Client("INGV"),
        waveform_cfg=ETNA_WAVEFORM_CONFIG,
        max_distance_km=30.0,
        channel_pattern="*HZ",
        probe_offset_hours=12,
        probe_duration_minutes=10,
        min_coverage_fraction=0.80,
        verbose=True,
        display_func=display,
    )

    station_screening_paths = export_station_screening_results(
        station_summary=station_summary,
        daily_probe=station_daily_probe,
        out_dir="../data/etna",
        waveform_cfg=ETNA_WAVEFORM_CONFIG,
        probe_offset_hours=12,
        probe_duration_minutes=10,
    )

    display(station_summary.head(20))

    for name, path in station_screening_paths.items():
        print(f"{name}: {path}")
else:
    print(
        "Station screening skipped. "
        "Set RUN_STATION_SCREENING = True to reproduce the ESLN selection."
    )

Discovered 562 vertical stream(s) before distance filtering.
Keeping 6 vertical stream(s) within 30 km of Etna summit.


c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\util.py:1123: UserWarning: Given string seems to not be a valid URI: '78'
  warnings.warn(msg)
c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\util.py:1123: UserWarning: Given string seems to not be a valid URI: '246'
  warnings.warn(msg)
c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\util.py:1123: UserWarning: Given string seems to not be a valid URI: '267'
  warnings.warn(msg)
c:\Users\cescedes\anaconda3\envs\M\Lib\site-packages\obspy\core\inventory\util.py:1123: UserWarning: Given string seems to not be a valid URI: '342'
  warnings.warn(msg)


,network,station,location,channel,metadata_start,metadata_end,metadata_start_date,metadata_end_date,lat,lon,elevation_m,distance_km
0,IV,ESLN,,BHZ,2005-07-10T00:00:00.000000Z,None,2005-07-10,open/unknown,37.69340,14.97440,1787.0,6.445278
1,IV,ESLN,,HHZ,2005-07-10T00:00:00.000000Z,None,2005-07-10,open/unknown,37.69340,14.97440,1787.0,6.445278
2,IV,ESLN,,LHZ,2005-07-10T00:00:00.000000Z,None,2005-07-10,open/unknown,37.69340,14.97440,1787.0,6.445278
3,IV,ESLN,,VHZ,2005-07-10T00:00:00.000000Z,None,2005-07-10,open/unknown,37.69340,14.97440,1787.0,6.445278
4,IV,ECTS,,HHZ,2007-07-18T12:00:00.000000Z,2019-12-23T11:00:00.000000Z,2007-07-18,2019-12-23,37.88250,15.12120,644.0,18.409270
5,IV,GIO,,SHZ,2007-12-04T12:00:00.000000Z,2013-03-13T00:00:00.000000Z,2007-12-04,2013-03-13,37.56667,15.10833,200.0,22.342209


[1/6] IV.ESLN..BHZ | distance=6.4 km | metadata period=2005-07-10 to open/unknown
[2/6] IV.ESLN..HHZ | distance=6.4 km | metadata period=2005-07-10 to open/unknown
[3/6] IV.ESLN..LHZ | distance=6.4 km | metadata period=2005-07-10 to open/unknown
[4/6] IV.ESLN..VHZ | distance=6.4 km | metadata period=2005-07-10 to open/unknown
[5/6] IV.ECTS..HHZ | distance=18.4 km | metadata period=2007-07-18 to 2019-12-23
[6/6] IV.GIO..SHZ | distance=22.3 km | metadata period=2007-12-04 to 2013-03-13


,network,station,location,channel,lat,lon,distance_km,metadata_start_date,metadata_end_date,n_days,...,successful_probe_end,successful_probe_start_date,successful_probe_end_date,successful_probe_ranges,missing_dates,mean_probe_coverage,min_probe_coverage,n_error_days,sample_rates,channel_rank
0,IV,ESLN,,HHZ,37.69340,14.97440,6.445278,2005-07-10,open/unknown,34,...,2008-05-15,2008-04-12,2008-05-15,2008-04-12 to 2008-05-15,,1.000000,1.000000,0,100.0,1
1,IV,ESLN,,BHZ,37.69340,14.97440,6.445278,2005-07-10,open/unknown,34,...,2008-05-15,2008-04-12,2008-05-15,2008-04-12 to 2008-05-15,,0.999990,0.999917,0,20.0,2
2,IV,ESLN,,LHZ,37.69340,14.97440,6.445278,2005-07-10,open/unknown,34,...,2008-05-15,2008-04-12,2008-05-15,2008-04-12 to 2008-05-15,,1.000000,1.000000,0,1.0,5
3,IV,ESLN,,VHZ,37.69340,14.97440,6.445278,2005-07-10,open/unknown,34,...,2008-05-15,2008-04-12,2008-05-15,2008-04-12 to 2008-05-15,,1.000000,1.000000,0,0.1,6
4,IV,ECTS,,HHZ,37.88250,15.12120,18.409270,2007-07-18,2019-12-23,34,...,2008-05-15,2008-04-12,2008-05-15,2008-04-12 to 2008-05-15,,0.994610,0.816733,0,100.0,1
5,IV,GIO,,SHZ,37.56667,15.10833,22.342209,2007-12-04,2013-03-13,34,...,2008-05-15,2008-04-12,2008-05-15,2008-04-12 to 2008-05-15,,0.997643,0.959967,0,50.0,4


report_txt: ..\data\etna\etna_station_screening_report.txt


## Hourly teleseismic waveform proxy

ESLN HHZ data are processed in padded daily chunks. The proxy is the hourly maximum of non-overlapping 120-second RMS values in the 0.03–0.30 Hz band.

In [3]:
waveform_df, waveform_failures = build_station_waveform_dataset(
    client=Client("INGV"),
    station=ETNA_WAVEFORM_CONFIG["station"],
    config=ETNA_WAVEFORM_CONFIG,
    cache_path="../data/etna/etna_esln_hhz_waveform.pkl",
    redownload=False, # True to re-download of waveform data, False to use cached data
)

print(f"Waveform rows: {len(waveform_df)}")
print(f"Failed daily chunks: {len(waveform_failures)}")
if waveform_failures:
    display(pd.DataFrame(waveform_failures, columns=["date", "error"]))

ESLN: loaded cached waveform features from ..\data\etna\etna_esln_hhz_waveform.pkl
Waveform rows: 816
Failed daily chunks: 0


## External variables

The retained external variables are soil CO₂ concentration and atmospheric pressure drop from ETNAGAS, plus hourly Open-Meteo precipitation. Isolated one-hour ETNAGAS gaps are linearly interpolated and reported explicitly.

Liuzzo, M., Giuffrida, G. B., & Gurrieri, S. (2025). *Etna CO2 Soil Flux during 2002–2010 (ECSF2002_2010)*. INGV. https://doi.org/10.13127/etna/ecsf2002_2010

Zippenfenig, P. (2023). *Open-Meteo.com Weather API*. Zenodo. https://doi.org/10.5281/ZENODO.7970649

In [4]:
etnagas_df, etnagas_interpolation_report = load_etnagas_csv(
    "../data/etna/3c.csv",
    value_cols=ETNA_GAS_METEO_COLS,
    start_time=ETNA_WAVEFORM_CONFIG["start"].isoformat(),
    end_time=ETNA_WAVEFORM_CONFIG["end"].isoformat(),
)

display(etnagas_interpolation_report)

,timestamp,interpolated_variables,method,maximum_gap
0,2008-05-06 13:00:00+00:00,"CO2_3, Patm_3",linear time interpolation,1 hour
1,2008-05-11 22:00:00+00:00,"CO2_3, Patm_3",linear time interpolation,1 hour
2,2008-05-13 13:00:00+00:00,"CO2_3, Patm_3",linear time interpolation,1 hour


In [6]:
weather_cache = Path("../data/etna/etna_openmeteo_hourly.csv")

if weather_cache.exists():
    weather_df = pd.read_csv(
        weather_cache,
        parse_dates=["timestamp"],
    )
    weather_df["timestamp"] = pd.to_datetime(
        weather_df["timestamp"],
        utc=True,
    )
else:
    weather_df = load_openmeteo_etna_weather(
        start_date="2008-04-12",
        end_date="2008-05-15",
    )
    weather_df.to_csv(weather_cache, index=False)

print(f"Weather rows: {len(weather_df)}")

Weather rows: 816


## Mt. Etna Seismic Catalogue 2000–2010

The catalogue supplies two hourly variables:

- `local_event_rate_state`: a past-only 48-hour local-seismicity state ending six hours before the current hour;
- `local_event_rate_response`: a positive six-hour event-count response relative to an earlier past baseline, used as the effect variable.

Alparone, S. C., Maiolino, V., Mostaccio, A., Scaltrito, A., Ursino, A., Barberi, G., et al. (2015). *Mt. Etna Seismic Catalog 2000–2010* [Data set]. INGV–Osservatorio Etneo. https://doi.org/10.13127/etnasc/2000_2010

In [ ]:
catalogue = load_etna_event_catalog_xls(
    "../data/etna/Etna catalogue_2000-2010.xls",
    quality_filter=False,
)

print(f"Catalogue rows: {len(catalogue)}")
print(
    "Catalogue range:",
    catalogue["timestamp"].min(),
    "to",
    catalogue["timestamp"].max(),
)

## Save the hourly dataset

All sources are joined by exact UTC hourly timestamps. The first rows lacking the required past-only catalogue windows are removed; no other missing rows are silently filled during the final merge.

In [ ]:
etna_dataset = create_etna_dataset(
    wave_df=waveform_df,
    start_time=ETNA_WAVEFORM_CONFIG["start"].isoformat(),
    end_time=ETNA_WAVEFORM_CONFIG["end"].isoformat(),
    catalog_df=catalogue,
    etnagas_df=etnagas_df,
    etnagas_cols=ETNA_GAS_METEO_COLS,
    weather_df=weather_df,
    weather_cols=ETNA_WEATHER_COLS,
    output_dir="../data/etna",
)

## Construction audit

This compact audit reports the final hourly-grid integrity, waveform failures, the leading rows removed for past-only proxy warm-up, and ETNAGAS interpolation count.

In [ ]:
expected_hours = int(
    (ETNA_WAVEFORM_CONFIG["end"] - ETNA_WAVEFORM_CONFIG["start"]) / 3600
)

audit = dataset_health_report(
    etna_dataset,
    "Etna canonical hourly dataset",
)
audit["waveform_failed_chunks"] = len(waveform_failures)
audit["proxy_warmup_rows_removed"] = expected_hours - len(etna_dataset)
audit["etnagas_interpolated_hours"] = len(
    etnagas_interpolation_report
)

display(audit)
display(distribution_summary(etna_dataset))

### Distribution diagnostics

Empirical density plots and summary statistics are used to inspect skewness, outliers, and scaling behavior. 


In [ ]:
plot_variable_pdfs(etna_dataset, filename="etna_pdf")

In [ ]:
summary_esln = distribution_summary(etna_dataset)
display(summary_esln)

### Log-log distribution diagnostics

These diagnostics use the canonical unscaled variables. True logarithmic axes require positive values, so zeros and negative observations are excluded variable by variable and reported explicitly.

In [ ]:
fig, axes, etna_loglog_report = plot_etna_loglog_distributions(
    dataframe=etna_dataset,
    save_dir="figures",
)

display(etna_loglog_report)

### Final overview figures

In [ ]:
plot_etna_thesis_figures(
    csv_path="../data/etna/etna_dataset.csv",
    event_time=EVENT_TIME,
    save_dir="figures",
    include_titles=False,
)

### Exact teleseismic-feature diagnostic

The following figures reuse the same response correction, frequency band, 120-second RMS windows, and hourly-maximum aggregation used in the canonical dataset.

In [ ]:
waveform_client = Client(
    base_url="https://webservices.ingv.it",
    service_mappings={
        "dataselect": "https://webservices.ingv.it/fdsnws/dataselect/1/",
        "station": "https://webservices.ingv.it/fdsnws/station/1/",
        "event": "https://webservices.ingv.it/fdsnws/event/1/",
    },
    _discover_services=False,
)

teleseismic_check = run_teleseismic_checks(
    client=waveform_client,
    station=ETNA_WAVEFORM_CONFIG["station"],
    config=ETNA_WAVEFORM_CONFIG,
    event_time=EVENT_TIME,
    save_dir="figures",
)

### Geographic Visualization

In [ ]:
figure, axis, source_table = plot_etna_all_variables_map(
    metadata=etna_observable_metadata(),
    satellite=True,
    save_dir="figures",
    filename="etna_map",
)

display(source_table)